# FedProx

> The implementation of FedProx client as in https://arxiv.org/pdf/2006.08848

In [ ]:
#| default_exp clients.fedprox

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import math
import os

from copy import deepcopy
from loguru import logger
from tqdm import tqdm

from fastcore.utils import patch
import torch
import torch.nn.functional as F

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

## FedProx  
Personalized Federated learning using Morseu-envelope

In [ ]:
#| export
@AlgorithmRegistry.register_client("fedprox")
class FedProxClient(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
        
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        
        self.global_params = deepcopy(list(self.model.parameters()))   

### Training

In [ ]:
#| export
@patch
def fit(self: FedProxClient):
    
    self.model = self.model.to(self.device)
    self.local_model = self.local_model.to(self.device)
    self.model.train()
    self.local_model.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            outputs = self.model(X)
            self.optimizer.zero_grad()
            loss = self.criterion(outputs, y)
            loss.backward()
            self.optimizer.step(self.global_params, self.device)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()